# NSW Bushfire Risk Mapping — Blue Mountains Case Study

This notebook builds a composite bushfire risk index for the Blue Mountains
region of NSW using Sentinel-2 vegetation/moisture indices and SRTM terrain
variables, then visualises it as an interactive map.

**Run this notebook in Google Colab** — it needs an authenticated Google
Earth Engine session, which Colab handles for you in the auth cell below.

Pipeline:
1. Authenticate & initialise Earth Engine
2. Define the area of interest (AOI) and time window
3. Build a cloud-free Sentinel-2 composite
4. Compute NDVI (fuel load) and NDMI (moisture)
5. Derive slope and solar-exposure from SRTM
6. Combine into a 0–100 risk index and 4-class risk map
7. Visualise interactively and export outputs for GitHub

## 0. Setup

In [ ]:
# If running in Colab, clone the repo so we can import src/ modules
import os
if not os.path.exists('nsw-bushfire-risk-mapping'):
    !git clone https://github.com/<your-username>/nsw-bushfire-risk-mapping.git
    %cd nsw-bushfire-risk-mapping

!pip install -q earthengine-api geemap folium

In [ ]:
import ee

# First time only: replace 'your-gee-project-id' with your GEE cloud project.
# Sign up for a free project at https://code.earthengine.google.com/
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')

In [ ]:
import sys
sys.path.append('src')

from risk_index import (
    mask_s2_clouds, compute_risk_index, DEFAULT_WEIGHTS
)
from mapping import build_risk_map

## 1. Area of interest: Blue Mountains, NSW

This region was severely affected by the 2019–2020 'Black Summer' fires,
making it a well-documented case study for validating the risk index.

In [ ]:
# Rough bounding box around the Blue Mountains / Katoomba area
aoi = ee.Geometry.Rectangle([150.10, -33.85, 150.45, -33.55])

center_lat, center_lon = -33.70, 150.28

## 2. Cloud-free Sentinel-2 composite

We use a spring/early-summer window (pre-fire-season) since dryness and
fuel load ahead of the fire season are the most operationally relevant
signals. Adjust `START_DATE` / `END_DATE` to test other seasons or years.

In [ ]:
START_DATE = '2019-10-01'
END_DATE = '2019-11-30'

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(mask_s2_clouds)
)

print('Number of scenes used:', s2_collection.size().getInfo())

s2_composite = s2_collection.median().clip(aoi)

## 3. Terrain data (SRTM 30m)

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)

## 4. Compute the composite risk index

In [ ]:
risk_image = compute_risk_index(s2_composite, dem, aoi, weights=DEFAULT_WEIGHTS)
print('Bands in output:', risk_image.bandNames().getInfo())

## 5. Interactive map

In [ ]:
m = build_risk_map(
    center_lat, center_lon, risk_image,
    ndvi_band=risk_image.select('NDVI'),
    ndmi_band=risk_image.select('NDMI'),
)
m

## 6. Export outputs for the GitHub repo

Saves a standalone HTML map (viewable via GitHub Pages / nbviewer) and a
static PNG preview for the README.

In [ ]:
m.save('outputs/blue_mountains_risk_map.html')
print('Saved interactive map to outputs/blue_mountains_risk_map.html')

## 7. (Optional) Sanity check against known fire history

The NSW RFS publishes historical fire extent polygons via NSW SEED
(https://www.seed.nsw.gov.au/). Overlaying the 2019-20 fire perimeter on
the risk map and checking that burnt areas skew toward 'High'/'Extreme'
classes is a good qualitative validation step and a strong talking point
in interviews — see `notebooks/02_validation.ipynb` (to be added) for a
worked example once you've downloaded the fire history shapefile.